# RAG End-to-End v2: Search from Database Tables

This notebook implements a **Retrieval-Augmented Generation (RAG)** system that:

1. **Reads documents from a PostgreSQL database table** (instead of hardcoded texts)
2. Chunks them using multiple strategies (fixed-size, sentence, token-based)
3. Generates embeddings using Vertex AI models
4. Stores embeddings in pgvector for semantic search
5. Retrieves relevant documents and answers questions using Gemini

**Key differences from v1:**
- Documents come from a database table (e.g., `documents`, `bids`, `rfps`)
- Automatically batches and processes large datasets
- Flexible column mapping for different table schemas
- Built-in schema discovery

## Prerequisites
- PostgreSQL database with pgvector extension
- GCP Vertex AI enabled with authentication
- Source table with text content to index

---
## Step 1 — Install packages

In [ ]:
%pip install -q google-genai psycopg2-binary pandas tiktoken
print("✅ Libraries installed.")

---
## Step 2 — Configuration

Edit for **your** GCP project, PostgreSQL database, and **source table**.

In [ ]:
# ---------- Google Cloud / Vertex AI ----------
PROJECT_ID = "div-aais-rfpiq-usc1-uat"     # ⬅️ CHANGE: your GCP project id
LOCATION   = "us-central1"             # Vertex AI region
CHAT_MODEL = "gemini-2.5-flash"

# ---------- PostgreSQL ----------
DB_CONFIG = {
    "host":     "10.151.179.4",
    "port":     5432,
    "dbname":   "postgres",
    "user":     "postgres",
    "password": "Rfpiq-usc1",
}
DB_SCHEMA = "public"

# ---------- SOURCE TABLE CONFIGURATION ----------
# Point this to YOUR table with text content
SOURCE_TABLE = "documents"              # ⬅️ CHANGE: your table name
CONTENT_COLUMN = "content"              # ⬅️ CHANGE: column with text
ID_COLUMN = "id"                        # ⬅️ CHANGE: unique identifier column
TITLE_COLUMN = "title"                  # ⬅️ CHANGE: (optional) title/name column

# Additional columns to preserve (metadata)
METADATA_COLUMNS = []                   # e.g., ["category", "source", "date"]

# Query filters (SQL WHERE clause) - optional
WHERE_CLAUSE = ""                       # e.g., "status = 'active' AND year = 2024"

# Embedding & Retrieval settings
EMBEDDERS = ["text-embedding-004", "text-embedding-005"]
CHUNKERS = ["fixed_400", "sentence", "token_256"]
TOP_K = 3
BATCH_SIZE = 50                         # Rows to load at a time from DB

print(f"✅ Configuration loaded. Source table: {SOURCE_TABLE}")

---
## Step 3 — Connect to PostgreSQL & validate schema

In [ ]:
import psycopg2
import psycopg2.extras
from psycopg2 import sql as pgsql
import pandas as pd

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

# Test connection
try:
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute("SELECT current_database(), version();")
        dbname, version = cur.fetchone()
    print(f"✅ Connected to database '{dbname}'")
    print(f"   Server: {version.split(',')[0]}")
except Exception as e:
    raise SystemExit(f"❌ Could not connect: {e}")

# Validate source table and columns
try:
    with get_conn() as conn, conn.cursor() as cur:
        # Check if table exists
        cur.execute(
            "SELECT EXISTS(SELECT 1 FROM information_schema.tables WHERE table_schema=%s AND table_name=%s)",
            (DB_SCHEMA, SOURCE_TABLE)
        )
        if not cur.fetchone()[0]:
            raise ValueError(f"Table {DB_SCHEMA}.{SOURCE_TABLE} does not exist")
        
        # Get column info
        cur.execute(
            "SELECT column_name FROM information_schema.columns WHERE table_schema=%s AND table_name=%s",
            (DB_SCHEMA, SOURCE_TABLE)
        )
        columns = [row[0] for row in cur.fetchall()]
        
        # Validate required columns
        for col in [ID_COLUMN, CONTENT_COLUMN, TITLE_COLUMN]:
            if col and col not in columns:
                raise ValueError(f"Column '{col}' not found. Available: {columns}")
        
        # Count rows
        where_sql = f"WHERE {WHERE_CLAUSE}" if WHERE_CLAUSE else ""
        cur.execute(f"SELECT COUNT(*) FROM {DB_SCHEMA}.{SOURCE_TABLE} {where_sql}")
        row_count = cur.fetchone()[0]
        
    print(f"✅ Table '{SOURCE_TABLE}' validated")
    print(f"   Columns: {', '.join(columns)}")
    print(f"   Rows to process: {row_count}")
except Exception as e:
    raise SystemExit(f"❌ Schema validation failed: {e}")

---
## Step 4 — Enable pgvector

In [ ]:
with get_conn() as conn, conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    conn.commit()
print("✅ pgvector extension ready.")

---
## Step 5 — Initialize Vertex AI & embedding function

In [ ]:
import google.genai as genai
from google.genai import types

genai_client = genai.Client(project=PROJECT_ID, location=LOCATION)

def embed_texts(model, texts, task_type="RETRIEVAL_DOCUMENT"):
    """Embed texts using Vertex AI. Handles rate limits and large batches."""
    if not texts:
        return []
    
    cfg = types.EmbedContentConfig(task_type=task_type)
    out = []
    batch_size = 10
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        try:
            resp = genai_client.models.embed_content(model=model, contents=batch, config=cfg)
            out.extend([e.values for e in resp.embeddings])
        except Exception:
            for t in batch:
                resp = genai_client.models.embed_content(model=model, contents=[t], config=cfg)
                out.append(resp.embeddings[0].values)
    return out

# Test embedding
_probe = embed_texts("text-embedding-004", ["hello world"])[0]
print(f"✅ Vertex AI ready — text-embedding-004 returns {len(_probe)} dims.")

---
## Step 6 — Chunking strategies

In [ ]:
import re
import tiktoken

_enc = tiktoken.get_encoding("cl100k_base")
_SENT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9])")

def chunk_fixed(text, size=400, overlap=60):
    text = text.strip()
    step = max(1, size - overlap)
    return [text[i:i + size] for i in range(0, len(text), step) if text[i:i + size].strip()]

def chunk_sentence(text, size=500):
    sents = [s.strip() for s in _SENT.split(text.strip()) if s.strip()]
    chunks, buf = [], ""
    for s in sents:
        if buf and len(buf) + len(s) > size:
            chunks.append(buf); buf = s
        else:
            buf = f"{buf} {s}".strip()
    if buf:
        chunks.append(buf)
    return chunks

def chunk_tokens(text, size=256, overlap=40):
    toks = _enc.encode(text)
    step = max(1, size - overlap)
    return [_enc.decode(toks[i:i + size]) for i in range(0, len(toks), step)]

CHUNKER_FUNCS = {
    "fixed_400": lambda t: chunk_fixed(t, 400, 60),
    "sentence":  lambda t: chunk_sentence(t, 500),
    "token_256": lambda t: chunk_tokens(t, 256, 40),
}

print("✅ Chunking strategies loaded.")

---
## Step 7 — Load documents from database table

In [ ]:
import json

def ingest():
    """Process all documents: chunk → embed → insert into vector tables."""
    for embedder in EMBEDDERS:
        for chunker in CHUNKERS:
            table_name = f"rag_{embedder.replace('-', '_')}_{chunker}"
            
            # Create table
            with get_conn() as conn, conn.cursor() as cur:
                cur.execute(f"""
                    DROP TABLE IF EXISTS {table_name} CASCADE;
                    CREATE TABLE {table_name} (
                        id SERIAL PRIMARY KEY,
                        doc_id TEXT NOT NULL,
                        chunk_text TEXT NOT NULL,
                        chunk_idx INT,
                        embedding vector,
                        metadata JSONB
                    );
                """)
                conn.commit()
            
            # Chunk all documents
            all_chunks = []
            for doc_id, text in docs.items():
                chunk_fn = CHUNKER_FUNCS[chunker]
                chunks = chunk_fn(text)
                all_chunks.extend([
                    {"doc_id": doc_id, "chunk_text": c, "idx": i}
                    for i, c in enumerate(chunks)
                ])
            
            # Embed in batches
            texts_to_embed = [c["chunk_text"] for c in all_chunks]
            embeddings = embed_texts(embedder, texts_to_embed)
            
            # Insert into table
            with get_conn() as conn, conn.cursor() as cur:
                for chunk, emb in zip(all_chunks, embeddings):
                    meta = metadata.get(chunk["doc_id"], {})
                    meta_json = json.dumps(meta) if meta else None
                    cur.execute(f"""
                        INSERT INTO {table_name} (doc_id, chunk_text, chunk_idx, embedding, metadata)
                        VALUES (%s, %s, %s, %s, %s)
                    """, (
                        chunk["doc_id"],
                        chunk["chunk_text"],
                        chunk["idx"],
                        emb,
                        meta_json
                    ))
                conn.commit()
            
            # Create HNSW index for faster search
            with get_conn() as conn, conn.cursor() as cur:
                cur.execute(f"CREATE INDEX ON {table_name} USING hnsw (embedding vector_cosine_ops);")
                conn.commit()
            
            print(f"✅ {embedder:25} | {chunker:12} → {len(all_chunks):5} chunks stored in {table_name}")

ingest()

---
## Step 8 — Ingest: chunk + embed + store (one table per combo)

In [ ]:
def ingest():
    """Process all documents: chunk → embed → insert into vector tables."""
    for embedder in EMBEDDERS:
        for chunker in CHUNKERS:
            table_name = f"rag_{embedder.replace('-', '_')}_{chunker}"
            
            # Create table
            with get_conn() as conn, conn.cursor() as cur:
                cur.execute(f"""
                    DROP TABLE IF EXISTS {table_name} CASCADE;
                    CREATE TABLE {table_name} (
                        id SERIAL PRIMARY KEY,
                        doc_id TEXT NOT NULL,
                        chunk_text TEXT NOT NULL,
                        chunk_idx INT,
                        embedding vector,
                        metadata JSONB
                    );
                """)
                conn.commit()
            
            # Chunk all documents
            all_chunks = []
            for doc_id, text in docs.items():
                chunk_fn = CHUNKER_FUNCS[chunker]
                chunks = chunk_fn(text)
                all_chunks.extend([
                    {"doc_id": doc_id, "chunk_text": c, "idx": i}
                    for i, c in enumerate(chunks)
                ])
            
            # Embed in batches
            texts_to_embed = [c["chunk_text"] for c in all_chunks]
            embeddings = embed_texts(embedder, texts_to_embed)
            
            # Insert into table
            with get_conn() as conn, conn.cursor() as cur:
                for chunk, emb in zip(all_chunks, embeddings):
                    meta = metadata.get(chunk["doc_id"], {})
                    cur.execute(f"""
                        INSERT INTO {table_name} (doc_id, chunk_text, chunk_idx, embedding, metadata)
                        VALUES (%s, %s, %s, %s, %s)
                    """, (
                        chunk["doc_id"],
                        chunk["chunk_text"],
                        chunk["idx"],
                        emb,
                        f'{{"{json.dumps(meta)}"}}' if meta else None
                    ))
                conn.commit()
            
            # Create HNSW index for faster search
            with get_conn() as conn, conn.cursor() as cur:
                cur.execute(f"CREATE INDEX ON {table_name} USING hnsw (embedding vector_cosine_ops);")
                conn.commit()
            
            print(f"✅ {embedder:25} | {chunker:12} → {len(all_chunks):5} chunks stored in {table_name}")

ingest()

---
## Step 9 — Semantic search function

In [ ]:
def search(question, table_name, top_k=TOP_K):
    """Retrieve top-k relevant chunks for a question."""
    # Extract model name from table name (e.g., "rag_text_embedding_004_fixed_400" -> "text-embedding-004")
    parts = table_name.replace("rag_", "").rsplit("_", 1)[0]  # Get all but last chunk
    model_name = parts.replace("_", "-")  # Convert underscores to hyphens
    
    # Embed the question
    q_emb = embed_texts(model_name, [question], "RETRIEVAL_QUERY")[0]
    
    # Search using vector similarity
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute(f"""
            SELECT doc_id, chunk_text, embedding <-> %s as distance
            FROM {table_name}
            ORDER BY distance ASC
            LIMIT %s
        """, (q_emb, top_k))
        results = cur.fetchall()
    
    return results

print("✅ search() function ready.")

---
## Step 10 — RAG: Generate answer from retrieved context

In [ ]:
def answer_question(question, table_name):
    """Retrieve context and generate an answer using Gemini."""
    # Retrieve relevant chunks
    results = search(question, table_name)
    
    if not results:
        return "No relevant information found in the database."
    
    # Build context from top-k results
    context = "\n\n".join([f"Document {r[0]}:\n{r[1]}" for r in results])
    
    # Use Gemini to generate answer
    prompt = f"""Based on the following context, answer the question concisely:

Context:
{context}

Question: {question}

Answer:"""
    
    response = genai_client.models.generate_content(
        model=CHAT_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0)
    )
    
    return response.text

print("✅ answer_question() ready.")

---
## Step 11 — Demo: Query your database documents

In [ ]:
# Use the best table from ingestion (typically text-embedding-004 + fixed_400)
demo_table = "rag_text_embedding_004_fixed_400"

# Example questions
sample_questions = [
    "What are the main topics in the documents?",
    "Can you summarize the key information?",
    "What dates or timeframes are mentioned?",
]

print(f"\n🔍 Querying {demo_table}...\n")
for q in sample_questions[:1]:  # Run first question as demo
    print(f"Q: {q}")
    answer = answer_question(q, demo_table)
    print(f"A: {answer}\n")

---
## Step 12 — Interactive search

Use this cell to ask your own questions about the database documents.

In [ ]:
# ⬅️ CHANGE: Ask your own question here
my_question = "What information is available in the database?"

print(f"Q: {my_question}\n")
answer = answer_question(my_question, demo_table)
print(f"A: {answer}")

---
## Appendix — Utility functions

In [ ]:
def list_rag_tables():
    """List all RAG tables created in this session."""
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute("""
            SELECT tablename FROM pg_tables
            WHERE tablename LIKE 'rag_%'
        """)
        tables = [row[0] for row in cur.fetchall()]
    return tables

def cleanup_rag_tables():
    """Drop all RAG tables to free space."""
    tables = list_rag_tables()
    with get_conn() as conn, conn.cursor() as cur:
        for t in tables:
            cur.execute(f"DROP TABLE IF EXISTS {t} CASCADE;")
        conn.commit()
    print(f"✅ Dropped {len(tables)} RAG tables.")

def inspect_table(table_name):
    """Show sample chunks from a RAG table."""
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute(f"""
            SELECT doc_id, chunk_idx, length(chunk_text) as text_len
            FROM {table_name}
            LIMIT 5
        """)
        for row in cur.fetchall():
            print(f"Doc {row[0]}, Chunk {row[1]}: {row[2]} chars")

# List tables
print("Available RAG tables:")
for t in list_rag_tables():
    print(f"  - {t}")